In [1]:
!nvidia-smi

%cd /content
!rm -rf CIRI-FS
!git clone --branch asal/CiriEXT4 https://github.com/isusbu/CIRI-FS.git
%cd /content/CIRI-FS

!git branch --show-current

Wed Jul 22 15:40:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -r requirements.txt
!pip install -q transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 999.8/999.8 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
!find "/content/drive/MyDrive" -maxdepth 6 \
    \( -iname "*cpd*" -o -iname "*ocd*" -o -iname "*ccd*" -o -iname "*ext4*" \) \
    | sort

/content/drive/MyDrive/CIRI_EXT4
/content/drive/MyDrive/Colab Notebooks/EXT4SD
/content/drive/MyDrive/Colab Notebooks/qwen_ext4_cpd_zero_shot


In [7]:
!ls -lah "/content/drive/MyDrive/CIRI_EXT4"

ls: /content/drive/MyDrive/CIRI_EXT4/Dataset: No such file or directory
total 4.5K
lrw------- 1 root root    0 Jul 22 03:59 Dataset -> /content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset
drwx------ 3 root root 4.0K Jul 13 05:28 Qwen2.5-Coder-7B-Instruct
-rw------- 1 root root  212 Jul 13 05:30 Qwen_zero_shot_metrics.txt


In [8]:
!readlink -f "/content/drive/MyDrive/CIRI_EXT4/Dataset"

/content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset


In [9]:
import os

DRIVE_DATA = os.path.realpath(
    "/content/drive/MyDrive/CIRI_EXT4/Dataset"
)

print(DRIVE_DATA)
print("Exists:", os.path.exists(DRIVE_DATA))

/content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset
Exists: True


In [12]:
DRIVE_DATA = "/content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset/EXT4_XML_Dataset"

In [14]:
!ls icse25_data/datasets/synthesize_config

alluxio  etcd	  ground_truth	hcommon  postgresql  yarn
django	 ext4_SD  hbase		hdfs	 redis	     zookeeper


In [15]:
!cp -r "$DRIVE_DATA/ext4_CPD" \
icse25_data/datasets/synthesize_config/

In [17]:
!cp "$DRIVE_DATA/Groundtruth/ext4_CPD.tsv" \
    icse25_data/datasets/synthesize_config/ground_truth/

In [18]:
!ls icse25_data/datasets/synthesize_config/ground_truth

alluxio.tsv  ext4_CPD.tsv  hcommon.tsv	   redis.tsv
django.tsv   ext4_SD.tsv   hdfs.tsv	   yarn.tsv
etcd.tsv     hbase.tsv	   postgresql.tsv  zookeeper.tsv


In [21]:
!echo "Correct:"
!find icse25_data/datasets/synthesize_config/ext4_CPD/correct -type f | wc -l

!echo "Erroneous:"
!find icse25_data/datasets/synthesize_config/ext4_CPD/erroneous -type f | wc -l

Correct:
7
Erroneous:
7


In [22]:
!python -m py_compile \
    ciri/ciri_eng.py \
    ciri/ciri_runner.py \
    ciri/query/llm_gen.py

In [23]:
!grep -n "\[Qwen Cost\]" ciri/query/llm_gen.py

262:            "[Qwen Cost] "


In [24]:
BENCHMARK = "ext4_CPD"
MODEL = "Qwen2.5-Coder-7B-Instruct"
MODE = "zero_shot"

OUTPUT_ROOT = (
    f"icse25_data/results/synthesize_config/"
    f"{BENCHMARK}/{MODEL}/{MODE}"
)

In [25]:
!rm -rf "$OUTPUT_ROOT"
!mkdir -p experiment_logs

In [26]:
!python -m ciri.ciri_eng \
  --input_path "icse25_data/datasets/synthesize_config/$BENCHMARK/erroneous" \
  --output_path "$OUTPUT_ROOT/erroneous" \
  --model "$MODEL" \
  --system ext4 \
  --version 1.47.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml \
  --verbose \
  2>&1 | tee "experiment_logs/${BENCHMARK}_erroneous.log"

2026-07-22 16:29:16 - Ciri - INFO - Using device: CUDA
2026-07-22 16:29:16 - Ciri - INFO - Using dtype: torch.bfloat16
Loading weights:   1%|          | 2/339 [00:08<24:52,  4.43s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 339/339 [01:02<00:00,  5.45it/s]
2026-07-22 16:33:22 - Ciri - INFO - Model loaded successfully on CUDA!
2026-07-22 16:33:22 - Ciri - INFO - [llm_gen] Using device: CUDA
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
2026-07-22 16:33:27 - Ciri - INFO - [Qwen Cost] call=1, input_tokens=294, output_tokens=25, total_tokens=319, 

In [27]:
!python -m ciri.ciri_eng \
  --input_path "icse25_data/datasets/synthesize_config/$BENCHMARK/correct" \
  --output_path "$OUTPUT_ROOT/correct" \
  --model "$MODEL" \
  --system ext4 \
  --version 1.47.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml \
  --verbose \
  2>&1 | tee "experiment_logs/${BENCHMARK}_correct.log"

2026-07-22 16:39:14 - Ciri - INFO - Using device: CUDA
2026-07-22 16:39:14 - Ciri - INFO - Using dtype: torch.bfloat16
Loading weights:   1%|          | 2/339 [00:08<24:05,  4.29s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 339/339 [01:01<00:00,  5.53it/s]
2026-07-22 16:40:22 - Ciri - INFO - Model loaded successfully on CUDA!
2026-07-22 16:40:22 - Ciri - INFO - [llm_gen] Using device: CUDA
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
2026-07-22 16:40:27 - Ciri - INFO - [Qwen Cost] call=1, input_tokens=294, output_tokens=25, total_tokens=319, 

In [28]:
!echo "Correct results:"
!find "$OUTPUT_ROOT/correct" -type f | wc -l

!echo "Erroneous results:"
!find "$OUTPUT_ROOT/erroneous" -type f | wc -l

Correct results:
7
Erroneous results:
7


In [29]:
!python icse25_data/script/result_parser.py \
  --project "$BENCHMARK" \
  --model "$MODEL" \
  --mode "$MODE" \
  | tee "${BENCHMARK}_metrics.txt"

[Ciri Result] on ext4_CPD with Qwen2.5-Coder-7B-Instruct and zero_shot mode
File-Level: Precision: N.A., Recall: 0.00, Accuracy: 0.50, F1: N.A.
Param-Level: Precision: N.A., Recall: 0.00, Accuracy: 0.93, F1: N.A.
